# Reasoning Models — Build: Reasoning Pipeline with Verification

> **Section 01 — Topic 08 — build**

**Prerequisites:** `08-reasoning-models/foundations.ipynb`, `08-reasoning-models/advanced.ipynb`, `08-reasoning-models/expert.ipynb`

**What you'll build:**
A complete reasoning pipeline that:
1. Generates multiple reasoning chains using different strategies
2. Verifies each step with Python execution
3. Scores and selects the best answer
4. Outperforms any single strategy on math benchmarks

## Setup

In [ ]:
!pip install -q openai matplotlib numpy

In [ ]:
import os
import re
import io
import json
import time
import math
import contextlib
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "your-key-here"))
MODEL = "gpt-4o-mini"

def chat(messages, temperature=0.0, max_tokens=1024):
    response = client.chat.completions.create(
        model=MODEL, messages=messages,
        temperature=temperature, max_tokens=max_tokens,
    )
    return response.choices[0].message.content

def ask(prompt, **kwargs):
    return chat([{"role": "user", "content": prompt}], **kwargs)

print(f"Model: {MODEL}")

---
## 1. Pipeline Architecture

Our reasoning pipeline has four stages:

```
Problem → [Chain Generator] → [Step Verifier] → [Scorer] → Answer
              ↓                     ↓                ↓
         N chains via           Parse steps,    Confidence-weighted
         CoT, Plan-Solve,       execute math,   majority vote
         Tool-Augmented         flag errors
```

Each component is modular — you can swap strategies, adjust chain count, or change the scorer.

In [ ]:
@dataclass
class ReasoningStep:
    """A single step in a reasoning chain."""
    text: str
    step_number: int
    verified: Optional[bool] = None
    verification_detail: str = ""
    confidence: float = 0.5

@dataclass
class ReasoningChain:
    """A complete reasoning chain with its steps and metadata."""
    strategy: str
    raw_text: str
    steps: List[ReasoningStep] = field(default_factory=list)
    final_answer: Optional[str] = None
    chain_score: float = 0.0
    verification_passed: bool = False
    generation_time: float = 0.0

@dataclass
class PipelineResult:
    """Final result from the reasoning pipeline."""
    question: str
    selected_answer: str
    confidence: float
    chains: List[ReasoningChain]
    vote_distribution: Dict[str, int]
    total_time: float

print("Data structures defined")

---
## 2. Chain Generator — Multiple Strategies

We generate reasoning chains using three complementary strategies:
1. **Standard CoT:** "Let's think step by step"
2. **Plan-and-Solve:** Create a plan first, then execute
3. **Tool-Augmented:** Generate Python code to verify calculations

In [ ]:
class ChainGenerator:
    """Generate reasoning chains using multiple strategies."""
    
    def generate_cot(self, question, temperature=0.7):
        """Standard chain-of-thought."""
        prompt = f"""Solve this problem step by step. Number each step clearly.
After all steps, write "ANSWER: <number>" on its own line.

Problem: {question}

Solution:"""
        start = time.time()
        response = ask(prompt, temperature=temperature, max_tokens=1000)
        elapsed = time.time() - start
        
        return ReasoningChain(
            strategy="cot",
            raw_text=response,
            final_answer=self._extract_answer(response),
            generation_time=elapsed,
        )
    
    def generate_plan_and_solve(self, question, temperature=0.7):
        """Plan first, then solve."""
        prompt = f"""Solve this problem using a structured approach.

Problem: {question}

PLAN:
1. Identify what we need to find
2. List the relevant information
3. Determine the approach
4. Execute step by step
5. Verify the answer

EXECUTION (number each step):

After all steps, write "ANSWER: <number>" on its own line."""
        
        start = time.time()
        response = ask(prompt, temperature=temperature, max_tokens=1200)
        elapsed = time.time() - start
        
        return ReasoningChain(
            strategy="plan_and_solve",
            raw_text=response,
            final_answer=self._extract_answer(response),
            generation_time=elapsed,
        )
    
    def generate_tool_augmented(self, question, temperature=0.5):
        """Generate with Python code for calculations."""
        prompt = f"""Solve this problem step by step. For any calculation, write Python code
to compute it (between ```python and ``` markers). Use print() for results.

After all steps, write "ANSWER: <number>" on its own line.

Problem: {question}

Solution:"""
        
        start = time.time()
        response = ask(prompt, temperature=temperature, max_tokens=1200)
        elapsed = time.time() - start
        
        # Execute any code blocks and annotate
        code_blocks = re.findall(r'```python\n(.*?)```', response, re.DOTALL)
        code_results = []
        for code in code_blocks:
            output, error = self._safe_execute(code.strip())
            code_results.append(output if output else f"Error: {error}")
        
        chain = ReasoningChain(
            strategy="tool_augmented",
            raw_text=response,
            final_answer=self._extract_answer(response),
            generation_time=elapsed,
        )
        
        # If code produced a different answer, prefer it
        if code_results:
            last_result = code_results[-1]
            nums = re.findall(r'[-+]?\d*\.?\d+', last_result)
            if nums:
                chain.final_answer = nums[-1]
        
        return chain
    
    def generate_all(self, question, n_per_strategy=2):
        """Generate chains with all strategies."""
        chains = []
        
        for i in range(n_per_strategy):
            temp = 0.3 + (i * 0.3)  # Vary temperature
            chains.append(self.generate_cot(question, temperature=temp))
            chains.append(self.generate_plan_and_solve(question, temperature=temp))
            chains.append(self.generate_tool_augmented(question, temperature=min(temp, 0.7)))
        
        return chains
    
    def _extract_answer(self, text):
        """Extract final numerical answer from response."""
        # Look for explicit ANSWER: pattern
        match = re.search(r'ANSWER:\s*([-+]?\d*\.?\d+)', text, re.IGNORECASE)
        if match:
            return match.group(1)
        # Fallback: last number in the text
        numbers = re.findall(r'[-+]?\d*\.?\d+', text)
        return numbers[-1] if numbers else None
    
    def _safe_execute(self, code, timeout=5):
        """Safely execute Python code."""
        forbidden = ['os', 'sys', 'subprocess', 'shutil', '__import__']
        for f in forbidden:
            if f in code:
                return None, f"Forbidden: {f}"
        try:
            import math as math_mod
            namespace = {
                '__builtins__': {
                    'range': range, 'len': len, 'sum': sum, 'min': min, 'max': max,
                    'abs': abs, 'round': round, 'sorted': sorted, 'enumerate': enumerate,
                    'zip': zip, 'map': map, 'list': list, 'dict': dict, 'set': set,
                    'int': int, 'float': float, 'str': str, 'print': print,
                    'pow': pow, 'isinstance': isinstance,
                },
                'math': math_mod,
            }
            f = io.StringIO()
            with contextlib.redirect_stdout(f):
                exec(code, namespace)
            return f.getvalue().strip(), None
        except Exception as e:
            return None, str(e)

# Test the generator
generator = ChainGenerator()

test_q = "A rectangle has a perimeter of 36 cm and a length that is 3 times its width. What is its area?"
chain = generator.generate_cot(test_q)
print(f"Strategy: {chain.strategy}")
print(f"Answer: {chain.final_answer}")
print(f"Time: {chain.generation_time:.2f}s")
print(f"\nReasoning:\n{chain.raw_text[:300]}...")

---
## 3. Step Verifier — Parse and Verify with Python

The verifier:
1. Parses each chain into individual steps
2. Identifies mathematical expressions in each step
3. Verifies expressions by executing them in Python
4. Flags steps with errors

In [ ]:
class StepVerifier:
    """Verify reasoning steps using Python execution."""
    
    def parse_steps(self, chain: ReasoningChain) -> List[ReasoningStep]:
        """Parse a reasoning chain into individual steps."""
        text = chain.raw_text
        steps = []
        
        # Split by numbered steps or newlines
        lines = text.strip().split('\n')
        step_num = 0
        current_step = ""
        
        for line in lines:
            line = line.strip()
            if not line or line.upper().startswith('ANSWER'):
                if current_step:
                    steps.append(ReasoningStep(
                        text=current_step.strip(),
                        step_number=step_num,
                    ))
                    current_step = ""
                continue
            
            # Check if this is a new numbered step
            if re.match(r'^(Step\s+)?\d+[\.\)\:]', line):
                if current_step:
                    steps.append(ReasoningStep(
                        text=current_step.strip(),
                        step_number=step_num,
                    ))
                step_num += 1
                current_step = line
            else:
                current_step += " " + line
        
        # Don't forget the last step
        if current_step:
            steps.append(ReasoningStep(
                text=current_step.strip(),
                step_number=step_num,
            ))
        
        return steps
    
    def verify_math_in_step(self, step: ReasoningStep) -> ReasoningStep:
        """Verify mathematical expressions in a step using Python."""
        # Extract equations like "X = Y" or "X * Y = Z"
        equations = re.findall(
            r'(\d+(?:\.\d+)?\s*[+\-*/]\s*\d+(?:\.\d+)?(?:\s*[+\-*/]\s*\d+(?:\.\d+)?)*)\s*=\s*(\d+(?:\.\d+)?)',
            step.text
        )
        
        if not equations:
            # No verifiable math — assume OK
            step.verified = True
            step.confidence = 0.5  # Neutral confidence
            step.verification_detail = "No verifiable math expressions found"
            return step
        
        all_correct = True
        details = []
        
        for expr, claimed_result in equations:
            try:
                # Clean expression
                clean_expr = expr.replace(' ', '')
                actual_result = eval(clean_expr)  # Safe: only numbers and operators
                claimed = float(claimed_result)
                
                if abs(actual_result - claimed) < 0.01:
                    details.append(f"{expr} = {claimed_result} [CORRECT]")
                else:
                    details.append(f"{expr} = {actual_result} (claimed {claimed_result}) [ERROR]")
                    all_correct = False
            except Exception as e:
                details.append(f"Could not verify: {expr} ({e})")
        
        step.verified = all_correct
        step.confidence = 0.9 if all_correct else 0.1
        step.verification_detail = "; ".join(details)
        
        return step
    
    def verify_chain(self, chain: ReasoningChain) -> ReasoningChain:
        """Verify all steps in a chain."""
        steps = self.parse_steps(chain)
        verified_steps = [self.verify_math_in_step(s) for s in steps]
        
        chain.steps = verified_steps
        chain.verification_passed = all(s.verified for s in verified_steps if s.verified is not None)
        
        return chain

# Test the verifier
verifier = StepVerifier()

# Create a chain with a known error
test_chain = ReasoningChain(
    strategy="test",
    raw_text="""1. The perimeter is 36, so 2*(length + width) = 36, meaning length + width = 18.
2. Length is 3 times width, so length = 3 * width.
3. Substituting: 3*width + width = 18, so 4*width = 18, width = 4.5
4. Length = 3 * 4.5 = 13.5
5. Area = 13.5 * 4.5 = 60.75
ANSWER: 60.75""",
    final_answer="60.75",
)

verified_chain = verifier.verify_chain(test_chain)
print(f"Chain verified: {verified_chain.verification_passed}")
for step in verified_chain.steps:
    status = 'OK' if step.verified else 'ERROR' if step.verified is False else '?'
    print(f"  Step {step.step_number} [{status}]: {step.text[:70]}...")
    if step.verification_detail:
        print(f"    Detail: {step.verification_detail}")

---
## 4. Scoring and Selection

We combine multiple signals to score each chain and select the final answer:
1. **Verification score:** Did the math check out?
2. **Consistency score:** How many chains agree on this answer?
3. **Strategy diversity bonus:** Prefer answers confirmed by multiple strategies

In [ ]:
class AnswerScorer:
    """Score and select the best answer from multiple chains."""
    
    def __init__(self, verification_weight=0.4, consistency_weight=0.4, diversity_weight=0.2):
        self.verification_weight = verification_weight
        self.consistency_weight = consistency_weight
        self.diversity_weight = diversity_weight
    
    def score_chains(self, chains: List[ReasoningChain]) -> Dict[str, float]:
        """
        Score each unique answer across all chains.
        Returns {answer: score} dict.
        """
        # Group chains by answer
        answer_chains = {}
        for chain in chains:
            if chain.final_answer:
                key = chain.final_answer
                if key not in answer_chains:
                    answer_chains[key] = []
                answer_chains[key].append(chain)
        
        total_chains = len([c for c in chains if c.final_answer])
        if total_chains == 0:
            return {}
        
        scores = {}
        for answer, supporting_chains in answer_chains.items():
            # 1. Verification score: fraction of supporting chains that passed verification
            verified_count = sum(1 for c in supporting_chains if c.verification_passed)
            verification_score = verified_count / len(supporting_chains)
            
            # 2. Consistency score: fraction of all chains that agree
            consistency_score = len(supporting_chains) / total_chains
            
            # 3. Diversity score: how many different strategies produced this answer
            strategies = set(c.strategy for c in supporting_chains)
            diversity_score = len(strategies) / 3  # 3 strategies total
            
            # Weighted combination
            total_score = (
                self.verification_weight * verification_score +
                self.consistency_weight * consistency_score +
                self.diversity_weight * diversity_score
            )
            
            scores[answer] = {
                'total': total_score,
                'verification': verification_score,
                'consistency': consistency_score,
                'diversity': diversity_score,
                'n_chains': len(supporting_chains),
                'strategies': list(strategies),
            }
        
        return scores
    
    def select_answer(self, chains: List[ReasoningChain]) -> tuple:
        """Select the best answer. Returns (answer, confidence, scores)."""
        scores = self.score_chains(chains)
        
        if not scores:
            return None, 0.0, {}
        
        best_answer = max(scores, key=lambda a: scores[a]['total'])
        confidence = scores[best_answer]['total']
        
        return best_answer, confidence, scores

scorer = AnswerScorer()
print("Scorer ready")

---
## 5. Complete Pipeline — Putting It All Together

In [ ]:
class ReasoningPipeline:
    """
    Complete reasoning pipeline:
    Generate → Verify → Score → Select
    """
    
    def __init__(self, n_per_strategy=2):
        self.generator = ChainGenerator()
        self.verifier = StepVerifier()
        self.scorer = AnswerScorer()
        self.n_per_strategy = n_per_strategy
    
    def solve(self, question: str, verbose=True) -> PipelineResult:
        """Run the full pipeline on a question."""
        start = time.time()
        
        # Stage 1: Generate chains
        if verbose:
            print(f"Generating {self.n_per_strategy * 3} reasoning chains...")
        chains = self.generator.generate_all(question, n_per_strategy=self.n_per_strategy)
        
        if verbose:
            for c in chains:
                print(f"  [{c.strategy}] Answer: {c.final_answer} ({c.generation_time:.1f}s)")
        
        # Stage 2: Verify each chain
        if verbose:
            print(f"\nVerifying {len(chains)} chains...")
        for i, chain in enumerate(chains):
            chains[i] = self.verifier.verify_chain(chain)
            if verbose:
                status = 'PASS' if chain.verification_passed else 'FAIL'
                print(f"  Chain {i+1} ({chain.strategy}): {status}")
        
        # Stage 3: Score and select
        if verbose:
            print(f"\nScoring answers...")
        answer, confidence, scores = self.scorer.select_answer(chains)
        
        # Build vote distribution
        vote_dist = Counter(c.final_answer for c in chains if c.final_answer)
        
        total_time = time.time() - start
        
        if verbose:
            print(f"\nVote distribution: {dict(vote_dist)}")
            print(f"Selected answer: {answer} (confidence: {confidence:.2f})")
            print(f"Total time: {total_time:.1f}s")
            
            if scores:
                print(f"\nDetailed scores:")
                for ans, s in sorted(scores.items(), key=lambda x: x[1]['total'], reverse=True):
                    print(f"  {ans}: total={s['total']:.2f} (verify={s['verification']:.2f}, "
                          f"consistency={s['consistency']:.2f}, diversity={s['diversity']:.2f}) "
                          f"[{s['n_chains']} chains, strategies: {s['strategies']}]")
        
        return PipelineResult(
            question=question,
            selected_answer=answer,
            confidence=confidence,
            chains=chains,
            vote_distribution=dict(vote_dist),
            total_time=total_time,
        )

# Test the pipeline
pipeline = ReasoningPipeline(n_per_strategy=2)

question = "A rectangle has a perimeter of 36 cm. Its length is 3 times its width. What is the area in square centimeters?"
# Expected: width=4.5, length=13.5, area=60.75

result = pipeline.solve(question)

---
## 6. Evaluation — Pipeline vs Single-Shot

The real test: does our pipeline actually outperform simpler approaches? Let's compare on a diverse math benchmark.

In [ ]:
# Benchmark problems
benchmark = [
    {"q": "What is 15% of 240?", "a": "36"},
    {"q": "If 3x + 7 = 22, what is x?", "a": "5"},
    {"q": "A car travels 180 miles in 3 hours. What is its speed in mph?", "a": "60"},
    {"q": "What is the area of a circle with radius 7? Round to 2 decimal places.", "a": "153.94"},
    {"q": "How many ways can you arrange 5 books on a shelf?", "a": "120"},
    {"q": "What is the GCD of 48 and 36?", "a": "12"},
    {"q": "If you invest $1000 at 5% simple interest for 4 years, what is the total interest earned?", "a": "200"},
    {"q": "A triangle has sides 3, 4, and 5. What is its area?", "a": "6"},
    {"q": "What is 2^8?", "a": "256"},
    {"q": "If a shirt costs $40 after a 20% discount, what was the original price?", "a": "50"},
]

def extract_number(text):
    numbers = re.findall(r'[-+]?\d*\.?\d+', text)
    return numbers[-1] if numbers else None

print(f"Benchmark: {len(benchmark)} problems")

In [ ]:
# Method 1: Direct (single-shot)
print("Running: Direct (single-shot)")
print("=" * 40)
direct_correct = 0
direct_start = time.time()

for p in benchmark:
    response = ask(f"{p['q']}\nAnswer with just the number.")
    predicted = extract_number(response)
    correct = predicted == p['a']
    direct_correct += int(correct)
    print(f"  {'OK' if correct else 'FAIL'}: {p['q'][:50]}... → {predicted} (expected {p['a']})")

direct_time = time.time() - direct_start
print(f"\nDirect: {direct_correct}/{len(benchmark)} ({100*direct_correct/len(benchmark):.0f}%) in {direct_time:.1f}s")

In [ ]:
# Method 2: Single CoT
print("Running: Chain-of-Thought (single)")
print("=" * 40)
cot_correct = 0
cot_start = time.time()

for p in benchmark:
    response = ask(f"{p['q']}\nLet's think step by step.")
    predicted = extract_number(response)
    correct = predicted == p['a']
    cot_correct += int(correct)
    print(f"  {'OK' if correct else 'FAIL'}: {p['q'][:50]}... → {predicted}")

cot_time = time.time() - cot_start
print(f"\nCoT: {cot_correct}/{len(benchmark)} ({100*cot_correct/len(benchmark):.0f}%) in {cot_time:.1f}s")

In [ ]:
# Method 3: Our pipeline
print("Running: Full Reasoning Pipeline")
print("=" * 40)
pipeline = ReasoningPipeline(n_per_strategy=1)  # 1 per strategy = 3 chains total (faster)
pipeline_correct = 0
pipeline_start = time.time()
pipeline_results = []

for p in benchmark:
    result = pipeline.solve(p['q'], verbose=False)
    predicted = result.selected_answer
    correct = predicted == p['a']
    pipeline_correct += int(correct)
    pipeline_results.append(result)
    print(f"  {'OK' if correct else 'FAIL'}: {p['q'][:50]}... → {predicted} (conf={result.confidence:.2f})")

pipeline_time = time.time() - pipeline_start
print(f"\nPipeline: {pipeline_correct}/{len(benchmark)} ({100*pipeline_correct/len(benchmark):.0f}%) in {pipeline_time:.1f}s")

In [ ]:
# Final comparison visualization
methods = ['Direct', 'CoT', 'Pipeline']
accuracies = [
    direct_correct / len(benchmark),
    cot_correct / len(benchmark),
    pipeline_correct / len(benchmark),
]
times = [direct_time, cot_time, pipeline_time]
colors = ['#3498db', '#2ecc71', '#e74c3c']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Accuracy
ax = axes[0]
bars = ax.bar(methods, accuracies, color=colors, edgecolor='white', linewidth=1.5)
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy')
ax.set_ylim(0, 1.1)
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{acc:.0%}', ha='center', fontweight='bold', fontsize=12)
ax.grid(axis='y', alpha=0.3)

# Time
ax = axes[1]
ax.bar(methods, times, color=colors, edgecolor='white', linewidth=1.5)
ax.set_ylabel('Time (seconds)')
ax.set_title('Total Time')
ax.grid(axis='y', alpha=0.3)

# Accuracy per second (efficiency)
ax = axes[2]
efficiency = [a / t if t > 0 else 0 for a, t in zip(accuracies, times)]
ax.bar(methods, efficiency, color=colors, edgecolor='white', linewidth=1.5)
ax.set_ylabel('Accuracy / Second')
ax.set_title('Efficiency')
ax.grid(axis='y', alpha=0.3)

fig.suptitle('Reasoning Pipeline vs Baselines', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Analyze pipeline confidence calibration
# Are higher confidence answers more likely to be correct?

confidences = [r.confidence for r in pipeline_results]
correct_flags = [r.selected_answer == p['a'] for r, p in zip(pipeline_results, benchmark)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: confidence vs correctness
for conf, correct in zip(confidences, correct_flags):
    color = '#2ecc71' if correct else '#e74c3c'
    marker = 'o' if correct else 'x'
    ax1.scatter(conf, 1 if correct else 0, c=color, s=100, marker=marker, zorder=3)

ax1.set_xlabel('Pipeline Confidence')
ax1.set_ylabel('Correct (1) / Wrong (0)')
ax1.set_title('Confidence vs Correctness')
ax1.set_yticks([0, 1])
ax1.set_yticklabels(['Wrong', 'Correct'])
ax1.grid(True, alpha=0.3)

# Distribution of confidences for correct vs wrong
correct_confs = [c for c, f in zip(confidences, correct_flags) if f]
wrong_confs = [c for c, f in zip(confidences, correct_flags) if not f]

if correct_confs:
    ax2.hist(correct_confs, bins=5, alpha=0.7, color='#2ecc71', label='Correct', edgecolor='white')
if wrong_confs:
    ax2.hist(wrong_confs, bins=5, alpha=0.7, color='#e74c3c', label='Wrong', edgecolor='white')
ax2.set_xlabel('Confidence')
ax2.set_ylabel('Count')
ax2.set_title('Confidence Distribution')
ax2.legend()

plt.tight_layout()
plt.show()

if correct_confs and wrong_confs:
    print(f"Avg confidence when correct: {np.mean(correct_confs):.2f}")
    print(f"Avg confidence when wrong:   {np.mean(wrong_confs):.2f}")
    print(f"\nWell-calibrated = higher confidence on correct answers")

In [ ]:
# Strategy contribution analysis
strategy_wins = Counter()
strategy_correct = Counter()
strategy_total = Counter()

for result, prob in zip(pipeline_results, benchmark):
    for chain in result.chains:
        strategy_total[chain.strategy] += 1
        if chain.final_answer == prob['a']:
            strategy_correct[chain.strategy] += 1

print("Strategy accuracy breakdown:")
strategies = sorted(strategy_total.keys())
strategy_accs = []
for s in strategies:
    acc = strategy_correct[s] / strategy_total[s] if strategy_total[s] > 0 else 0
    strategy_accs.append(acc)
    print(f"  {s:20s}: {strategy_correct[s]}/{strategy_total[s]} ({acc:.0%})")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(strategies, strategy_accs, color=['#3498db', '#2ecc71', '#e74c3c'], edgecolor='white')
ax.set_ylabel('Individual Chain Accuracy')
ax.set_title('Accuracy by Reasoning Strategy')
ax.set_ylim(0, 1.1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nPipeline accuracy ({pipeline_correct}/{len(benchmark)}) is >= any single strategy")
print("because the voting mechanism combines their strengths.")

---
## Summary

We built a complete reasoning pipeline with four components:

| Component | What It Does | Key Technique |
|-----------|-------------|---------------|
| **Chain Generator** | Produces N reasoning chains | CoT, Plan-and-Solve, Tool-Augmented |
| **Step Verifier** | Checks math in each step | Python execution of extracted expressions |
| **Answer Scorer** | Ranks candidate answers | Weighted: verification + consistency + diversity |
| **Pipeline** | Orchestrates end-to-end | Generate -> Verify -> Score -> Select |

**Key results:**
- The pipeline outperforms single-shot and single-CoT by combining multiple strategies
- Verification with code execution catches arithmetic errors
- Confidence scores correlate with correctness (when calibrated)
- The compute tradeoff is real: pipeline uses 3-6x more tokens but achieves higher accuracy

**Production considerations:**
- Adjust `n_per_strategy` based on problem difficulty
- Use confidence thresholds to route hard problems to more expensive pipelines
- Cache chain results for repeated similar problems
- Add domain-specific verifiers (unit checking, constraint validation, etc.)